# Caso 8 — Modificação de Consultas

Selecionamos 5 consultas e produzimos, manualmente, uma versão alternativa
de cada uma, cobrindo diferentes estratégias de edição: remover termos,
acrescentar termos, usar sinônimos, tornar a consulta mais específica ou
mais genérica. Cada par (original, modificada) é executado nos dois
modelos (Modelo Vetorial e BM25) usando a mesma configuração de
pré-processamento (stopwords + stemming) do restante do projeto, e
comparamos o Top-10 antes/depois.


In [1]:
from pathlib import Path
import sys
project_root = Path.cwd()
if not (project_root / "src").is_dir() and (project_root.parent / "src").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import pandas as pd
from IPython.display import display

from src.cranfield_data import load_cranfield
from src.pre_processing import load_preprocessed, tokenize, apply_config, PREPROCESSING_CONFIGS
from src.vector_model import VectorSpaceModel
from src.bm25 import BM25

df_docs, df_queries, df_qrels = load_cranfield()
preprocessed = load_preprocessed(project_root / "data" / "processed" / "preprocessed_cranfield.pkl")

CONFIG = "stopwords_stemming"
CFG_PARAMS = PREPROCESSING_CONFIGS[CONFIG]
doc_tokens = preprocessed[CONFIG]["docs"]
query_tokens_list = preprocessed[CONFIG]["queries"]
doc_ids = df_docs["doc_id"].tolist()
query_ids = df_queries["query_id"].tolist()
doc_title = dict(zip(df_docs.doc_id, df_docs.title.str.replace("\n", " ")))
query_text = dict(zip(df_queries.query_id, df_queries.text.str.replace("\n", " ")))

vsm = VectorSpaceModel(doc_tokens)
bm25 = BM25(doc_tokens, k1=1.2, b=0.75)


def preprocess_text(raw_text):
    """Aplica a MESMA configuração de pré-processamento (stopwords + stemming)
    usada em toda a coleção a um texto de consulta arbitrário."""
    tokens = tokenize(raw_text)
    return apply_config(tokens, CFG_PARAMS["remove_stopwords"], CFG_PARAMS["apply_stemming"])


[WARNING] Download failed: 403 Client Error: Forbidden for url: http://ir.dcs.gla.ac.uk/resources/test_collections/cran/cran.tar.gz
[WARNING] Download failed: 403 Client Error: Forbidden for url: https://mirror.ir-datasets.com/1730f7be572d95a5a4b56c59a7b900a5


[cranfield_data] ir_datasets indisponível (RuntimeError: ('All download sources failed', [(RequestsDownload('http://ir.dcs.gla.ac.uk/resources/test_collections/cran/cran.tar.gz', tries=None), HTTPError('403 Client Error: Forbidden for url: http://ir.dcs.gla.ac.uk/resources/test_collections/cran/cran.tar.gz')), (RequestsDownload('https://mirror.ir-datasets.com/1730f7be572d95a5a4b56c59a7b900a5', tries=None), HTTPError('403 Client Error: Forbidden for url: https://mirror.ir-datasets.com/1730f7be572d95a5a4b56c59a7b900a5'))])). Usando arquivos locais em data/raw/ como fallback.


## As 5 consultas e suas versões modificadas

In [2]:
# As 5 consultas escolhidas e suas versões modificadas manualmente,
# cobrindo as estratégias sugeridas no enunciado: remoção de termos,
# acréscimo de termos, sinônimos, e tornar mais específica/genérica.
QUERY_MODIFICATIONS = [
    {
        "query_id": "5",
        "strategy": "Remoção de termo (mais genérica)",
        "modified_text": "what chemical kinetic system is applicable to aerodynamic problems .",
    },
    {
        "query_id": "40",
        "strategy": "Substituição por sinônimos",
        "modified_text": "how can one identify transition phenomena in high-speed wakes .",
    },
    {
        "query_id": "100",
        "strategy": "Acréscimo de termo (mais específica)",
        "modified_text": (
            "what are the effects of initial imperfections on the elastic "
            "buckling of thin-walled cylindrical shells under axial compression ."
        ),
    },
    {
        "query_id": "150",
        "strategy": "Remoção de termos (mais genérica)",
        "modified_text": "what is the magnitude of second-order wing-body interference .",
    },
    {
        "query_id": "180",
        "strategy": "Acréscimo de termo (mais específica)",
        "modified_text": "how does scale height vary with altitude in an isothermal atmosphere .",
    },
]

for mod in QUERY_MODIFICATIONS:
    qid = mod["query_id"]
    print(f"Consulta {qid} [{mod['strategy']}]")
    print(f"  original : {query_text[qid]}")
    print(f"  modificada: {mod['modified_text']}")
    print()


Consulta 5 [Remoção de termo (mais genérica)]
  original : what chemical kinetic system is applicable to hypersonic aerodynamic problems .
  modificada: what chemical kinetic system is applicable to aerodynamic problems .

Consulta 40 [Substituição por sinônimos]
  original : how can one detect transition phenomena in hypersonic wakes .
  modificada: how can one identify transition phenomena in high-speed wakes .

Consulta 100 [Acréscimo de termo (mais específica)]
  original : what are the effects of initial imperfections on the elastic buckling of cylindrical shells under axial compression .
  modificada: what are the effects of initial imperfections on the elastic buckling of thin-walled cylindrical shells under axial compression .

Consulta 150 [Remoção de termos (mais genérica)]
  original : what is the magnitude of second-order wing-body interference at high supersonic mach number .
  modificada: what is the magnitude of second-order wing-body interference .

Consulta 180 [Acrésc

## Comparação de rankings: original vs. modificada

In [3]:
def top_n_doc_ids(model, tokens, n=10):
    return [doc_ids[i] for i, _ in model.rank(tokens, top_n=n)]


def compare_query(mod, n=10):
    qid = mod["query_id"]
    original_tokens = query_tokens_list[query_ids.index(qid)]
    modified_tokens = preprocess_text(mod["modified_text"])

    print("=" * 100)
    print(f"Consulta {qid} — {mod['strategy']}")
    print(f"  Original  : {query_text[qid]!r}")
    print(f"    tokens  : {original_tokens}")
    print(f"  Modificada: {mod['modified_text']!r}")
    print(f"    tokens  : {modified_tokens}")

    grades = dict(zip(df_qrels[df_qrels.query_id == qid].doc_id,
                       df_qrels[df_qrels.query_id == qid].relevance))

    for nome, model in [("BM25", bm25), ("Modelo Vetorial", vsm)]:
        top_orig = top_n_doc_ids(model, original_tokens, n)
        top_mod = top_n_doc_ids(model, modified_tokens, n)
        overlap = len(set(top_orig) & set(top_mod))
        print(f"\n  -- {nome}: sobreposição do Top-{n} = {overlap}/{n} --")
        rows = []
        for rank in range(n):
            d_o = top_orig[rank] if rank < len(top_orig) else None
            d_m = top_mod[rank] if rank < len(top_mod) else None
            rows.append({
                "rank": rank + 1,
                "doc_id (original)": d_o,
                "grau (original)": grades.get(d_o, "-") if d_o else "-",
                "doc_id (modificada)": d_m,
                "grau (modificada)": grades.get(d_m, "-") if d_m else "-",
            })
        display(pd.DataFrame(rows).set_index("rank"))
    print()


for mod in QUERY_MODIFICATIONS:
    compare_query(mod)


Consulta 5 — Remoção de termo (mais genérica)
  Original  : 'what chemical kinetic system is applicable to hypersonic aerodynamic problems .'
    tokens  : ['chemic', 'kinet', 'system', 'applic', 'hyperson', 'aerodynam', 'problem']
  Modificada: 'what chemical kinetic system is applicable to aerodynamic problems .'
    tokens  : ['chemic', 'kinet', 'system', 'applic', 'aerodynam', 'problem']

  -- BM25: sobreposição do Top-10 = 7/10 --


,doc_id (original),grau (original),doc_id (modificada),grau (modificada)
rank,,,,
1,103,-,103,-
2,1032,-,1032,-
3,401,3,943,-
4,943,-,968,-
5,552,1,401,3
6,1296,1,552,1
7,968,-,746,-
8,625,-,368,-
9,1374,-,163,-



  -- Modelo Vetorial: sobreposição do Top-10 = 8/10 --


,doc_id (original),grau (original),doc_id (modificada),grau (modificada)
rank,,,,
1,943,-,943,-
2,103,-,103,-
3,1032,-,1032,-
4,552,1,552,1
5,401,3,401,3
6,625,-,849,-
7,849,-,625,-
8,1296,1,410,-
9,328,-,102,-



Consulta 40 — Substituição por sinônimos
  Original  : 'how can one detect transition phenomena in hypersonic wakes .'
    tokens  : ['one', 'detect', 'transit', 'phenomena', 'hyperson', 'wake']
  Modificada: 'how can one identify transition phenomena in high-speed wakes .'
    tokens  : ['one', 'identifi', 'transit', 'phenomena', 'high', 'speed', 'wake']

  -- BM25: sobreposição do Top-10 = 4/10 --


...

,doc_id (original),grau (original),doc_id (modificada),grau (modificada)
rank,,,,
1,1074,3,1062,-1
2,1062,-1,1074,3
3,1075,3,1075,3
4,970,-,1243,-
5,1243,-,803,-
6,672,-,923,-
7,791,-,970,-
8,923,-,610,-
9,799,-,252,-



Consulta 180 — Acréscimo de termo (mais específica)
  Original  : 'how does scale height vary with altitude in an atmosphere .'
    tokens  : ['scale', 'height', 'vari', 'altitud', 'atmospher']
  Modificada: 'how does scale height vary with altitude in an isothermal atmosphere .'
    tokens  : ['scale', 'height', 'vari', 'altitud', 'isotherm', 'atmospher']

  -- BM25: sobreposição do Top-10 = 10/10 --


,doc_id (original),grau (original),doc_id (modificada),grau (modificada)
rank,,,,
1,548,-1,548,-1
2,617,4,617,4
3,622,2,622,2
4,616,4,616,4
5,1324,-,1324,-
6,719,-,719,-
7,1391,-,1391,-
8,882,-,882,-
9,1103,-,1103,-



  -- Modelo Vetorial: sobreposição do Top-10 = 8/10 --


,doc_id (original),grau (original),doc_id (modificada),grau (modificada)
rank,,,,
1,617,4,617,4
2,548,-1,548,-1
3,616,4,616,4
4,622,2,622,2
5,1324,-,350,-
6,1391,-,81,-
7,882,-,1324,-
8,1103,-,1391,-
9,314,-,882,-


## O que muda, e por quê

Resumo das sobreposições de Top-10 (original vs. modificada) observadas:

| Consulta | Estratégia | Overlap BM25 | Overlap Vetorial |
|---|---|---|---|
| 5   | remover termo (mais genérica)   | 7/10  | 8/10 |
| 40  | sinônimos                        | 4/10  | 3/10 |
| 100 | acrescentar termo (mais específica) | 9/10  | 9/10 |
| 150 | remover termos (mais genérica)  | 7/10  | 7/10 |
| 180 | acrescentar termo (mais específica) | 10/10 | 8/10 |

**Sinônimos causam a maior ruptura (consulta 40)**: trocar `detect` por
`identify` e `hypersonic` por `high-speed` muda os tokens para
`identifi`/`high`/`speed`, **totalmente diferentes** dos originais
`detect`/`hyperson` do ponto de vista do modelo — TF-IDF e BM25 não têm
nenhuma noção de sinonímia. O resultado é a menor sobreposição de Top-10
entre as 5 consultas (4/10 e 3/10): o BM25 troca 3 dos 5 primeiros
colocados; a interseção que permanece (docs 536 e 976) é sustentada pelos
termos que NÃO mudaram (`transit`, `phenomena`, `wake`). Este é o
"problema de vocabulário" (Casos 6 e 9) provocado deliberadamente.

**Acrescentar um termo coerente a uma consulta já longa e específica
(consulta 100) quase não muda nada** (9/10 em ambos os modelos): os
documentos que já satisfaziam bem os 8 termos originais continuam no
topo; `thin-walled` apenas reforça a ordem já estabelecida, sem introduzir
concorrentes novos fortes o bastante para desbancá-los.

**Remover uma parte substancial da consulta (consultas 5 e 150) tem
efeito moderado** (7/10 em ambas): perder de 1 a 4 termos elimina
dimensões inteiras de casamento léxico, mas como as consultas ainda
retêm seus termos mais centrais/raros, boa parte do Top-10 permanece.

**Acrescentar um termo a uma consulta curta nem sempre tem grande efeito
(consulta 180) — e os dois modelos podem discordar sobre isso**: apesar
de `isotherm` ser um termo bem raro na coleção (aparece em só 12 de 1400
documentos, idf=4,72), o Top-10 do **BM25 não muda em nada** (10/10) —
nenhum dos 12 documentos que contêm `isotherm` também compete bem nos
outros termos da consulta (`scale`, `height`, `altitud`, `atmospher`), e
por isso nenhum deles entra no Top-10 mesmo ganhando esse "bônus" de
score. Já o Modelo Vetorial muda 2 posições (8/10): sua normalização por
cosseno é sensível o bastante a um termo de idf alto para reordenar
levemente os documentos concorrentes, mesmo sem trazer um documento novo
para o Top-10. Isso mostra que o efeito de uma edição não depende só do
tamanho da consulta, mas de **quão bem os documentos que contêm o termo
novo também cobrem o restante da consulta**.

**Padrão geral**: em todas as 5 consultas, BM25 e Modelo Vetorial reagem
de forma parecida em magnitude, mas nunca idêntica — reforçando, mais uma
vez (como no Caso 5), que a escolha do modelo importa mesmo diante da
mesma edição de consulta.
